In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Employee_ETL") \
    .getOrCreate()



In [0]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/myvolume/data/employees.csv")

df.show()

+------+----+--------------+------+----+
|emp_id|name|         email|salary|dept|
+------+----+--------------+------+----+
|   101| Ram| ram@gmail.com| 50000|  IT|
|   102| Sam| sam@gmail.com| 60000|  HR|
|   103|John|john@gmail.com| 70000|  IT|
|   103|John|john@gmail.com| 70000|  IT|
+------+----+--------------+------+----+



In [0]:
df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- dept: string (nullable = true)



In [0]:
df = df.dropDuplicates()

In [0]:
df.show()

+------+----+--------------+------+----+
|emp_id|name|         email|salary|dept|
+------+----+--------------+------+----+
|   102| Sam| sam@gmail.com| 60000|  HR|
|   101| Ram| ram@gmail.com| 50000|  IT|
|   103|John|john@gmail.com| 70000|  IT|
+------+----+--------------+------+----+



In [0]:
from pyspark.sql.functions import col, count, when

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+------+----+-----+------+----+
|emp_id|name|email|salary|dept|
+------+----+-----+------+----+
|     0|   0|    0|     0|   0|
+------+----+-----+------+----+



In [0]:
df = df.fillna({
    "dept": "UNKNOWN",
    "salary": 0
})

df.show()

+------+----+--------------+------+----+
|emp_id|name|         email|salary|dept|
+------+----+--------------+------+----+
|   101| Ram| ram@gmail.com| 50000|  IT|
|   102| Sam| sam@gmail.com| 60000|  HR|
|   103|John|john@gmail.com| 70000|  IT|
|   103|John|john@gmail.com| 70000|  IT|
+------+----+--------------+------+----+



In [0]:
from pyspark.sql.functions import col

df = df.withColumn(
    "salary",
    col("salary").cast("double")
)

In [0]:
df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- salary: double (nullable = false)
 |-- dept: string (nullable = false)



In [0]:
it_df = df.filter(
    col("dept") == "IT"
).show()

+------+----+--------------+-------+----+
|emp_id|name|         email| salary|dept|
+------+----+--------------+-------+----+
|   101| Ram| ram@gmail.com|50000.0|  IT|
|   103|John|john@gmail.com|70000.0|  IT|
+------+----+--------------+-------+----+



In [0]:
from pyspark.sql.functions import lit

df = df.withColumn(
    "bonus",
    col("salary") * lit(0.10)  # lit used for constant value
)

df.show()

+------+----+--------------+------+----+------+
|emp_id|name|         email|salary|dept| bonus|
+------+----+--------------+------+----+------+
|   101| Ram| ram@gmail.com| 50000|  IT|5000.0|
|   102| Sam| sam@gmail.com| 60000|  HR|6000.0|
|   103|John|john@gmail.com| 70000|  IT|7000.0|
|   103|John|john@gmail.com| 70000|  IT|7000.0|
+------+----+--------------+------+----+------+



In [0]:
from pyspark.sql.functions import current_timestamp,current_date

df = df.withColumn(
    "load_timeStamp",
    current_timestamp()    
).withColumn(
    "load_date",
    current_date()

)

df.show()

+------+----+--------------+------+----+------+--------------------+----------+
|emp_id|name|         email|salary|dept| bonus|      load_timeStamp| load_date|
+------+----+--------------+------+----+------+--------------------+----------+
|   101| Ram| ram@gmail.com| 50000|  IT|5000.0|2026-06-10 10:10:...|2026-06-10|
|   102| Sam| sam@gmail.com| 60000|  HR|6000.0|2026-06-10 10:10:...|2026-06-10|
|   103|John|john@gmail.com| 70000|  IT|7000.0|2026-06-10 10:10:...|2026-06-10|
|   103|John|john@gmail.com| 70000|  IT|7000.0|2026-06-10 10:10:...|2026-06-10|
+------+----+--------------+------+----+------+--------------------+----------+



In [0]:
df = df.orderBy(
    col("salary").desc()
)
df.show()

+------+----+--------------+------+----+------+--------------------+----------+
|emp_id|name|         email|salary|dept| bonus|      load_timeStamp| load_date|
+------+----+--------------+------+----+------+--------------------+----------+
|   103|John|john@gmail.com| 70000|  IT|7000.0|2026-06-10 10:11:...|2026-06-10|
|   103|John|john@gmail.com| 70000|  IT|7000.0|2026-06-10 10:11:...|2026-06-10|
|   102| Sam| sam@gmail.com| 60000|  HR|6000.0|2026-06-10 10:11:...|2026-06-10|
|   101| Ram| ram@gmail.com| 50000|  IT|5000.0|2026-06-10 10:11:...|2026-06-10|
+------+----+--------------+------+----+------+--------------------+----------+



In [0]:
from pyspark.sql.functions import sum, min, max, mean

dept_salary = df.groupBy("dept") \
    .sum("salary").alias("total_salary") 
dept_salary.show()

+----+-----------+
|dept|sum(salary)|
+----+-----------+
|  IT|     190000|
|  HR|      60000|
+----+-----------+



In [0]:
from pyspark.sql.functions import sum, min, max, mean

dept_salary = df.groupBy("dept") \
    .agg(
        sum("salary").alias("total_salary"),
        min("salary").alias("min_salary"),
        max("salary").alias("max_salary"),
        mean("salary").alias("avg_salary")
    )

dept_salary.show()

+----+------------+----------+----------+------------------+
|dept|total_salary|min_salary|max_salary|        avg_salary|
+----+------------+----------+----------+------------------+
|  IT|      190000|     50000|     70000|63333.333333333336|
|  HR|       60000|     60000|     60000|           60000.0|
+----+------------+----------+----------+------------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy(col("salary").desc())

df_rank = df.withColumn(
    "rn",
    row_number().over(window_spec)
)

top_emp = df_rank.filter(
    col("rn") == 2
)

top_emp.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+----+--------------+------+----+------+--------------------+----------+---+
|emp_id|name|         email|salary|dept| bonus|      load_timeStamp| load_date| rn|
+------+----+--------------+------+----+------+--------------------+----------+---+
|   103|John|john@gmail.com| 70000|  IT|7000.0|2026-06-10 10:20:...|2026-06-10|  2|
+------+----+--------------+------+----+------+--------------------+----------+---+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, dense_rank

window_spec = Window.orderBy(col("salary").desc())

df_rank = df.withColumn(
    "dr",
    dense_rank().over(window_spec)
)

top_emp = df_rank.filter(col("dr") == 2)

top_emp.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+----+-------------+------+----+------+--------------------+----------+---+
|emp_id|name|        email|salary|dept| bonus|      load_timeStamp| load_date| dr|
+------+----+-------------+------+----+------+--------------------+----------+---+
|   102| Sam|sam@gmail.com| 60000|  HR|6000.0|2026-06-10 10:21:...|2026-06-10|  2|
+------+----+-------------+------+----+------+--------------------+----------+---+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("dept") \
                    .orderBy(col("salary").desc())

df_rank = df.withColumn(
    "rn",
    row_number().over(window_spec)
)

top_emp = df_rank.filter(
    col("rn") == 1
)

top_emp.show()

+------+----+--------------+-------+----+------+--------------------+---+
|emp_id|name|         email| salary|dept| bonus|           load_date| rn|
+------+----+--------------+-------+----+------+--------------------+---+
|   102| Sam| sam@gmail.com|60000.0|  HR|6000.0|2026-06-10 00:25:...|  1|
|   103|John|john@gmail.com|70000.0|  IT|7000.0|2026-06-10 00:25:...|  1|
+------+----+--------------+-------+----+------+--------------------+---+



In [0]:
dept_df = spark.createDataFrame([
    (1, "IT"),
    (2, "HR"),
    (3, "FINANCE")
], ["dept_id", "dept"])

final_df = df.join(
    dept_df,
    "dept",
    "left"
)
df.show()
final_df.show()

+------+----+--------------+-------+----+------+--------------------+
|emp_id|name|         email| salary|dept| bonus|           load_date|
+------+----+--------------+-------+----+------+--------------------+
|   103|John|john@gmail.com|70000.0|  IT|7000.0|2026-06-10 00:28:...|
|   102| Sam| sam@gmail.com|60000.0|  HR|6000.0|2026-06-10 00:28:...|
|   101| Ram| ram@gmail.com|50000.0|  IT|5000.0|2026-06-10 00:28:...|
+------+----+--------------+-------+----+------+--------------------+

+----+------+----+--------------+-------+------+--------------------+-------+
|dept|emp_id|name|         email| salary| bonus|           load_date|dept_id|
+----+------+----+--------------+-------+------+--------------------+-------+
|  HR|   102| Sam| sam@gmail.com|60000.0|6000.0|2026-06-10 00:28:...|      2|
|  IT|   101| Ram| ram@gmail.com|50000.0|5000.0|2026-06-10 00:28:...|      1|
|  IT|   103|John|john@gmail.com|70000.0|7000.0|2026-06-10 00:28:...|      1|
+----+------+----+--------------+-------+

In [0]:
final_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/myvolume/output/employees")

In [0]:
from pyspark.sql.functions import max

last_loaded_date = "2026-06-10"

incremental_df = df.filter(
    col("load_date") > last_loaded_date
)

incremental_df.show()

+------+----+--------------+-------+----+------+--------------------+
|emp_id|name|         email| salary|dept| bonus|           load_date|
+------+----+--------------+-------+----+------+--------------------+
|   103|John|john@gmail.com|70000.0|  IT|7000.0|2026-06-10 00:34:...|
|   102| Sam| sam@gmail.com|60000.0|  HR|6000.0|2026-06-10 00:34:...|
|   101| Ram| ram@gmail.com|50000.0|  IT|5000.0|2026-06-10 00:34:...|
+------+----+--------------+-------+----+------+--------------------+



In [0]:
record_count = df.count()

print(f"Source Record Count: {record_count}")

Source Record Count: 3


In [0]:
jdbc_url = "jdbc:oracle:thin:@//localhost:1521/xe"

oracle_df = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "EMPLOYEES") \
    .option("user", "hr") \
    .option("password", "hr") \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .load()

oracle_df.show()

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-5984361051844945>, line 11
      1 jdbc_url = "jdbc:oracle:thin:@//localhost:1521/xe"
      3 oracle_df = spark.read.format("jdbc") \
      4     .option("url", jdbc_url) \
      5     .option("dbtable", "EMPLOYEES") \
   (...)
      8     .option("driver", "oracle.jdbc.OracleDriver") \
      9     .load()
---> 11 oracle_df.show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         rai